<a href="https://colab.research.google.com/github/patriciarrs/RAG/blob/main/PatriciaSilva_RAG_FinalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Final Project Information - RAG [don't edit this section]

## IMPORTANT
**Add comments to your code whenever necessary to explain your logic.**

## Submission
- Single Deliverable: Google Colab Notebook
- File Name Format: [YourName]_RAG_FinalProject.ipynb (optional but it helps us track your file)
- Submit as: ZIP file containing only the .ipynb file (the student portal doesn't accpet .ipynb files)

## Mandatory
- Ingestion Phase
- Inference Phase

# 2. Project Specific Information

Add all the information you find useful to explain your project, the steps you took, the features implemented, and possible enhancements for the future.

## Project Description
[2-3 sentences explaining what your RAG system does and what documents it works with]

## Steps
1. Data collection and preprocessing
2. ...

## Features Implemented
- [x] RAG v2 baseline (mandatory)
- [ ] < additional feature >

# 3. Installation

In [18]:
%pip install "langchain==0.3.27" -qqq
%pip install "langchain-community==0.3.31" -qqq
%pip install "langchain-openai==0.3.35" -qqq
%pip install "langchain-pinecone==0.2.0" -qqq
%pip install pypdf -qqq
%pip install flashrank -qqq
%pip install ragas -qqq

!pip install -q unstructured layoutparser torchvision

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
gradio 5.50.0 requires aiofiles<25.0,>=22.0, but you have aiofiles 25.1.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ja

# 4. Setup

In [19]:
def load_secrets(keys):
    import os

    try:
        from google.colab import userdata  # type: ignore
        values = {key: userdata.get(key) for key in keys}
        in_colab = True
    except Exception:
        # Load from .env file for local development
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            raise ImportError(
                "python-dotenv is required for local development. "
                "Install it with: pip install python-dotenv"
            )

        # Get values from environment (loaded from .env file)
        values = {key: os.getenv(key) for key in keys}
        in_colab = False

    missing = [key for key, value in values.items() if not value]
    if missing:
        if in_colab:
            raise ValueError(f"Missing keys: {', '.join(missing)}. Set them in Colab Secrets.")
        else:
            env_file_help = (
                f"Missing keys: {', '.join(missing)}.\n\n"
                "Create a .env file in the project root with:\n"
                + "\n".join([f"{key}=YOUR_VALUE_HERE" for key in missing])
            )
            raise ValueError(env_file_help)

    for key, value in values.items():
        os.environ[key] = value

    return values

In [20]:
try:
    secrets = load_secrets(['PINECONE_API_KEY', 'OPENAI_API_KEY'])
except ValueError as e:
    print(e)

# 5. Global Variables

In [43]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain.text_splitter import RecursiveCharacterTextSplitter

INDEX_NAME = "rag"
NAMESPACE = "run_1" # Change this for different runs

# Embeddings Model
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1536
)

# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini"
)

# Classification LLM
classification_llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0
)

# Vectorstore with Namespace support
vectorstore = PineconeVectorStore(
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace=NAMESPACE
)

# Parent splitter (large chunks for context)
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,   # Large chunks for rich context in answers
    chunk_overlap=400, # 20% overlap
)

# Child splitter (small chunks for search)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,   # Small chunks for precise search matching
    chunk_overlap=75, # 15% overlap
)

# InMemoryStore for parent documents
parent_docstore = InMemoryStore()

# ParentDocumentRetriever
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=parent_docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

# 6. Indexing / Ingestion Phase

## Load documents (data) Function

In [23]:
import requests
from langchain_community.document_loaders import UnstructuredURLLoader

def load_public_github_markdown_dir(owner, repo, path, branch="main"):
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={branch}"
    response = requests.get(api_url)

    if response.status_code != 200:
        raise Exception(f"Failed to fetch directory contents. Check your repository details or path. Status: {response.status_code}")

    contents = response.json()

    raw_urls = [
        item["download_url"]
        for item in contents
        if item["type"] == "file" and item["name"].endswith(".md")
    ]

    if not raw_urls:
        print("No markdown files found in the specified directory.")
        return []

    print(f"Found {len(raw_urls)} markdown files. Downloading and parsing...")

    loader = UnstructuredURLLoader(urls=raw_urls)
    documents = loader.load()

    return documents

## Load documents

In [25]:
print("Loading files...")

REPO_OWNER = "patriciarrs"
REPO_NAME = "RAG"
DIR_PATH = "knowledge_base"

documents = load_public_github_markdown_dir(REPO_OWNER, REPO_NAME, DIR_PATH)

print(f"Successfully loaded {len(documents)} documents into memory!")

Loading files...
Found 20 markdown files. Downloading and parsing...
Successfully loaded 20 documents into memory!


## Topic Detection Function

In [31]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def detect_document_topic(document) -> str:
    """
    DETECT DOCUMENT TOPIC

    Args:
        documents (list): List of Document objects from PyPDFLoader

    Returns:
        str: Detected topic (e.g., 'runbooks', 'ADRs')
    """

    topic_detection_template = ChatPromptTemplate.from_template(
      """
      Analyze the following document content and determine its primary topic.

      Document content:
      {content}

      Based on this content, what is the primary topic? Answer with a single word or short phrase (e.g., 'runbooks', 'ADRs').

      Examples:
      If the document is about Engineering Runbooks, answer: runbooks
      If the document is about Architecture Decision Records, answer: ADRs
      If the document is about Onboarding, answer: Onboarding
      If the document is about Company Policies & Standards, answer: Policies & Standards
      If the document is about System Reference, answer: System Reference

      Primary topic:
      """
    )

    topic_detection_chain = topic_detection_template | classification_llm | StrOutputParser()

    # Extract sample content from first few pages
    sample_content = ""
    sample_content += document.page_content + "\n\n"

    # Limit to 4000 characters to save costs
    sample_content = sample_content[:4000]

    detected_topic = topic_detection_chain.invoke({
        "content": sample_content
    }).strip().lower()

    return detected_topic

## Clean Text Function

In [27]:
import re

def clean_markdown_for_rag(text: str) -> str:
    # 1. Strip YAML frontmatter (text between the first pair of --- at the start)
    text = re.sub(r'^---\s*\n.*?\n---\s*\n', '', text, flags=re.DOTALL)

    # 2. Clean Markdown Images: Keep the alt text description, drop the URL noise
    # Transforms ![Alt Text](http://url.com/image.png) -> Image: Alt Text
    text = re.sub(r'!\[(.*?)\]\((.*?)\)', r'Image: \1', text)

    # 3. Clean Hyperlinks: Keep the anchor text, drop or preserve the URL cleanly
    # Transforms [Anchor Text](http://url.com) -> Anchor Text
    text = re.sub(r'\[(.*?)\]\((.*?)\)', r'\1', text)

    # 4. Remove raw inline HTML tags (e.g., <br>, <div class="...">)
    text = re.sub(r'<[^>]*>', '', text)

    # 5. Collapse excessive consecutive blank lines down to a maximum of two
    # (Preserves paragraph breaks for the structural text splitters)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()

## Ingest Function

In [44]:
def ingest_document(document):
    # Auto-detect topic
    print(f"\n[1/4] Auto-detecting topic...")
    detected_topic = detect_document_topic(document)
    print(f"✓ Topic: '{detected_topic}'")

    # Preprocessing
    print(f"\n[2/4] Preprocessing...")
    document.page_content = clean_markdown_for_rag(document.page_content)
    print(f"✓ Cleaned document content")

    # Add metadata
    print(f"\n[3/4] Adding metadata...")
    document.metadata.update({'topic': detected_topic})
    print(f"✓ Metadata added")

    # Store in vectorstore
    print(f"\n[4/4] Storing in vectorstore (Namespace: {NAMESPACE})...")

    # 1. ParentDocumentRetriever
    # The underlying vectorstore already has the namespace configured
    parent_retriever.add_documents([document])
    print(f"  ✓ Stored in parent-child vectorstore")

    # 2. Regular chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
    )
    chunks = splitter.split_documents([document])

    # Adding specifically to the namespaced vectorstore
    vectorstore.add_documents(chunks)
    print(f"  ✓ Stored in main vectorstore ({len(chunks)} chunks)")

    print(f"\n✓ Ingestion complete!")
    return 1, detected_topic

In [42]:
def ingest_documents():
    """
    INGESTION FOR EVALUATION

    Stores documents in vectorstore
    """

    total_docs = len(documents)
    print("-" * 80)
    print(f"STARTING INGESTION OF {total_docs} DOCUMENTS")
    print("-" * 80)

    for i, document in enumerate(documents, 1):
      print(f">>> Processing document {i} of {total_docs}...")
      ingest_document(document)
      print(f">>> Finished document {i} of {total_docs}.")

## Ingest

In [41]:
ingest_documents()

--------------------------------------------------------------------------------
STARTING INGESTION
--------------------------------------------------------------------------------

[1/4] Auto-detecting topic...
✓ Topic: 'architecture decision records (adrs)'

[2/4] Preprocessing...
✓ Cleaned document content

[3/4] Adding metadata...
✓ Metadata added

[4/4] Storing in vectorstores...
  ✓ Stored in parent-child vectorstore
  ✓ Stored in main vectorstore (9 chunks)

✓ Ingestion complete!

[1/4] Auto-detecting topic...
✓ Topic: 'architecture decision records (adrs)'

[2/4] Preprocessing...
✓ Cleaned document content

[3/4] Adding metadata...
✓ Metadata added

[4/4] Storing in vectorstores...
  ✓ Stored in parent-child vectorstore
  ✓ Stored in main vectorstore (9 chunks)

✓ Ingestion complete!

[1/4] Auto-detecting topic...
✓ Topic: 'architecture decision records (adrs)'

[2/4] Preprocessing...
✓ Cleaned document content

[3/4] Adding metadata...
✓ Metadata added

[4/4] Storing in vector

# 7. Inference Phase (RAG)

In [ ]:
# =========================================
# Example Steps:
# =========================================
# Retriever setup (if not declared during setup)
# Prompt template
# LLM integration
# LCEL chain creation

# 8. User Interface (Gradio or other) - optional

In [ ]:
# =========================================
# Example Steps:
# =========================================
# Gradio UI implementation
# Query processing function
# Launch interface

# 9. Testing / Demo

In [ ]:
# =========================================
# Example Steps:
# =========================================
# Example Queries to Test
# Functions to Test
# Demo information